# Implémentation Qiskit — Grover avec distribution initiale arbitraire

L'idée de l'article est simple : au lieu de commencer Grover avec une superposition uniforme, on part d'un état initial quelconque :

\[|\psi_0\rangle = \sum_{x=0}^{N-1} a_x |x\rangle\]

Puis on applique les mêmes blocs que dans Grover :

1. oracle : changement de phase des états marqués ;
2. diffusion : inversion autour de la moyenne ;
3. mesure.

La différence avec le notebook du qubit auxiliaire est qu'ici **on n'ajoute pas de qubit auxiliaire** et on ne cherche pas à corriger la probabilité initiale avec une rotation \(R_y(\phi)\). On étudie directement ce que devient Grover si les amplitudes initiales sont non uniformes.


## 1. Imports

Si Qiskit n'est pas installé :

```bash
pip install qiskit qiskit-aer matplotlib numpy
```


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit.library import MCMTGate, ZGate
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram


## 2. Paramètres du problème

On choisit :

- `n` : nombre de qubits de recherche ;
- `N = 2**n` : nombre total d'états ;
- `marked_states` : liste des états marqués.

Exemple : avec `n = 4`, l'espace contient \(N=16\) états. On cherche ici l'état `1011`.


In [ ]:
marked_states = ["1011"]
n = len(marked_states[0])
N = 2**n
r = len(marked_states)

print("Nombre de qubits n =", n)
print("Nombre total d'états N =", N)
print("Nombre d'états marqués r =", r)
print("États marqués =", marked_states)


## 3. Préparation d'une distribution initiale arbitraire

Dans Grover standard, on utiliserait :

```python
qc.h(range(n))
```

ce qui donne une distribution uniforme.

Ici, on crée volontairement un état **non uniforme**. Pour que l'exemple reste lisible, on part d'une superposition presque uniforme, puis on ajoute une petite perturbation. Cela correspond bien à l'esprit de l'article : l'état initial peut être bruité ou imparfait.


In [ ]:
def normalize(amplitudes):
    amplitudes = np.asarray(amplitudes, dtype=complex)
    norm = np.linalg.norm(amplitudes)
    if norm == 0:
        raise ValueError("Le vecteur d'amplitudes ne peut pas être nul.")
    return amplitudes / norm

rng = np.random.default_rng(seed=7)

# Base uniforme + perturbation réelle : état non uniforme mais pas trop défavorable
uniform = np.ones(N, dtype=complex) / np.sqrt(N)
noise = 0.10 * rng.normal(size=N)
amplitudes = normalize(uniform + noise)

# Variante possible : état complexe arbitraire
# amplitudes = normalize(rng.normal(size=N) + 1j * rng.normal(size=N))

print("Somme des probabilités =", np.sum(np.abs(amplitudes)**2))
print("Probabilités initiales :")
for i, amp in enumerate(amplitudes):
    bitstring = format(i, f"0{n}b")
    print(bitstring, " -> ", round(abs(amp)**2, 4))


## 4. Fonctions : oracle et diffuseur de Grover

On garde les deux briques classiques de Grover :

- l'oracle multiplie les états marqués par \(-1\) ;
- le diffuseur réalise l'inversion autour de la moyenne.

C'est exactement ce que l'article conserve après avoir supprimé l'initialisation uniforme.


In [ ]:
def oracle(marked_states):
    """
    Oracle de phase : multiplie les états marqués par -1.

    Les états marqués sont donnés comme chaînes binaires, par exemple "1011".
    """
    if not isinstance(marked_states, list):
        marked_states = [marked_states]

    num_qubits = len(marked_states[0])
    qc = QuantumCircuit(num_qubits, name="Oracle")

    for target in marked_states:
        # Convention Qiskit : on inverse la chaîne pour agir sur les bons qubits
        rev_target = target[::-1]
        zero_inds = [i for i, bit in enumerate(rev_target) if bit == "0"]

        qc.x(zero_inds)

        if num_qubits == 1:
            qc.z(0)
        else:
            qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)

        qc.x(zero_inds)

    return qc


def diffusion(num_qubits):
    """
    Diffuseur de Grover : inversion autour de la moyenne.
    """
    qc = QuantumCircuit(num_qubits, name="Diffusion")

    qc.h(range(num_qubits))
    qc.x(range(num_qubits))

    if num_qubits == 1:
        qc.z(0)
    else:
        qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)

    qc.x(range(num_qubits))
    qc.h(range(num_qubits))

    return qc


## 5. Théorie de l'article : évolution des amplitudes moyennes

L'article sépare les amplitudes en deux familles :

- \(k_i(t)\) pour les états marqués ;
- \(l_i(t)\) pour les états non marqués.

On calcule leurs moyennes initiales :

\[
\bar{k}(0)=\frac{1}{r}\sum_{i \in marked} k_i(0)
\]

\[
\bar{l}(0)=\frac{1}{N-r}\sum_{i \notin marked} l_i(0)
\]

Puis l'article montre que l'évolution entière est contrôlée par ces moyennes, avec :

\[
\cos(\omega)=1-\frac{2r}{N}
\]

On code ci-dessous la solution exacte donnée par l'article.


In [ ]:
def marked_indices_from_strings(marked_states):
    return [int(s, 2) for s in marked_states]


def biham_exact_amplitudes(initial_amplitudes, marked_states, t):
    """
    Solution exacte de l'article après t itérations de Grover généralisé.

    Retourne le vecteur complet des amplitudes théoriques.
    """
    initial_amplitudes = np.asarray(initial_amplitudes, dtype=complex)
    N = len(initial_amplitudes)
    marked = marked_indices_from_strings(marked_states)
    r = len(marked)

    if not (1 <= r < N):
        raise ValueError("Il faut au moins un état marqué et au moins un état non marqué.")

    unmarked = [i for i in range(N) if i not in marked]

    k0 = initial_amplitudes[marked]
    l0 = initial_amplitudes[unmarked]

    kbar0 = np.mean(k0)
    lbar0 = np.mean(l0)

    beta = np.sqrt(r / (N - r))
    omega = np.arccos(1 - 2 * r / N)

    f1 = lbar0 + 1j * beta * kbar0
    f2 = lbar0 - 1j * beta * kbar0

    F1 = np.exp(1j * omega * t) * f1
    F2 = np.exp(-1j * omega * t) * f2

    lbar_t = 0.5 * (F1 + F2)
    kbar_t = (F1 - F2) / (2j * beta)

    # Les écarts à la moyenne sont des constantes de mouvement
    delta_k = k0 - kbar0
    delta_l = l0 - lbar0

    kt = kbar_t + delta_k
    lt = lbar_t + ((-1) ** t) * delta_l

    out = np.zeros(N, dtype=complex)
    out[marked] = kt
    out[unmarked] = lt
    return out


def success_probability(amplitudes, marked_states):
    marked = marked_indices_from_strings(marked_states)
    return float(np.sum(np.abs(np.asarray(amplitudes)[marked])**2))


def theory_probabilities(initial_amplitudes, marked_states, max_iter):
    return [
        success_probability(biham_exact_amplitudes(initial_amplitudes, marked_states, t), marked_states)
        for t in range(max_iter + 1)
    ]


## 6. Choisir le nombre d'itérations

Dans Grover standard, on prend environ :

\[
T \approx \frac{\pi}{4}\sqrt{\frac{N}{r}}
\]

Dans l'article, le temps optimal dépend aussi de la phase liée à la distribution initiale. Pour une implémentation simple, on peut tester plusieurs valeurs de \(t\) avec la formule exacte, puis prendre celle qui maximise la probabilité de mesurer un état marqué.


In [ ]:
omega = np.arccos(1 - 2 * r / N)
max_iter = int(np.ceil(np.pi / omega)) + 2

probs_theory = theory_probabilities(amplitudes, marked_states, max_iter)
best_t = int(np.argmax(probs_theory))

print("Probabilité initiale de succès =", round(probs_theory[0], 4))
print("Meilleur nombre d'itérations trouvé =", best_t)
print("Probabilité théorique maximale sur la fenêtre =", round(probs_theory[best_t], 4))

plt.figure(figsize=(8, 4))
plt.plot(range(max_iter + 1), probs_theory, marker="o")
plt.xlabel("Nombre d'itérations de Grover")
plt.ylabel("Probabilité de mesurer un état marqué")
plt.title("Grover avec distribution initiale arbitraire — prédiction théorique")
plt.grid(True)
plt.show()


## 7. Construction du circuit Qiskit

Ici, contrairement à Grover standard, on ne fait pas :

```python
qc.h(range(n))
```

On utilise directement :

```python
qc.initialize(amplitudes, range(n))
```

pour préparer l'état initial arbitraire.


In [ ]:
def grover_arbitrary_initial_state(marked_states, initial_amplitudes, iterations, measure=True):
    """
    Circuit de Grover généralisé :
    - état initial arbitraire ;
    - oracle de phase ;
    - diffuseur standard ;
    - mesure optionnelle.
    """
    initial_amplitudes = normalize(initial_amplitudes)
    n = int(np.log2(len(initial_amplitudes)))

    if 2**n != len(initial_amplitudes):
        raise ValueError("La taille du vecteur d'amplitudes doit être une puissance de 2.")

    if measure:
        qc = QuantumCircuit(n, n)
    else:
        qc = QuantumCircuit(n)

    qc.initialize(initial_amplitudes, range(n))

    for _ in range(iterations):
        qc.compose(oracle(marked_states), inplace=True)
        qc.compose(diffusion(n), inplace=True)

    if measure:
        qc.measure(range(n), range(n))

    return qc

qc_final = grover_arbitrary_initial_state(marked_states, amplitudes, best_t, measure=True)
qc_final.draw(output="mpl", style="iqp")


## 8. Simulation par vecteur d'état

Avant de faire une simulation avec des shots, on peut vérifier directement les amplitudes finales.


In [ ]:
qc_state = grover_arbitrary_initial_state(marked_states, amplitudes, best_t, measure=False)
state = Statevector.from_instruction(qc_state)

final_probs = state.probabilities_dict()
print("Probabilité finale des états :")
for state_label, prob in sorted(final_probs.items()):
    print(state_label, " -> ", round(prob, 4))

print("
Probabilité de succès finale =", round(success_probability(state.data, marked_states), 4))


## 9. Simulation avec mesures

On exécute maintenant le circuit comme dans une expérience simulée, avec un nombre fini de mesures.


In [ ]:
simulator = AerSimulator()
qc_compiled = qc_final.decompose().decompose()

job = simulator.run(qc_compiled, shots=2048)
result = job.result()
counts = result.get_counts()

print("Résultats :", counts)
plot_histogram(counts, title="Grover avec distribution initiale arbitraire")


## 10. Comparaison avec Grover standard

Pour voir la différence, on peut comparer avec l'état initial uniforme. Cette fois, les amplitudes sont toutes égales à \(1/\sqrt{N}\).


In [ ]:
uniform_amplitudes = np.ones(N, dtype=complex) / np.sqrt(N)

probs_uniform = theory_probabilities(uniform_amplitudes, marked_states, max_iter)

plt.figure(figsize=(8, 4))
plt.plot(range(max_iter + 1), probs_theory, marker="o", label="État initial arbitraire")
plt.plot(range(max_iter + 1), probs_uniform, marker="s", label="État initial uniforme")
plt.xlabel("Nombre d'itérations de Grover")
plt.ylabel("Probabilité de succès")
plt.title("Comparaison : distribution arbitraire vs uniforme")
plt.grid(True)
plt.legend()
plt.show()

print("Meilleur t arbitraire =", int(np.argmax(probs_theory)))
print("Meilleur t uniforme   =", int(np.argmax(probs_uniform)))
print("Pmax arbitraire =", round(max(probs_theory), 4))
print("Pmax uniforme   =", round(max(probs_uniform), 4))


## 11. Conclusion

Ce notebook implémente bien l'algorithme décrit dans l'article :

- on part d'une distribution initiale arbitraire ;
- on applique l'oracle de Grover ;
- on applique l'inversion autour de la moyenne ;
- on répète ces deux opérations ;
- on mesure.

La conclusion importante est que Grover peut encore amplifier les états marqués même si l'état initial n'est pas uniforme. En revanche, la probabilité maximale atteignable dépend de la distribution initiale : certaines distributions sont favorables, d'autres beaucoup moins.

À retenir pour ton projet : ce notebook est différent du notebook avec qubit auxiliaire. Ici, il n'y a pas de qubit en plus et pas de rotation \(R_y(\phi)\). On teste directement la robustesse de Grover face à un état initial non uniforme.
